# data ingestion from JDBC - microsoft SQL server driver

In [0]:
# previous code/old code - without using common utils functions and configurations


from common_utils.logging import logger

logger = get_logger('ds2b_sqlserver')

url = "jdbc:sqlserver://rivadata.database.windows.net:1433;databaseName=batch2;encrypt=true;trustServerCertificate=true;loginTimeout=90" 

user = "rivadata" 

password = dbutils.secrets.get(scope='retail-platform-dev', key='sqlserver_password')

dbtable = "retail.customers"

df = spark.read.format('jdbc')\
    .option('url',url)\
    .option('dbtable',dbtable)\
    .option('user', user)\
    .option('password', password)\
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')\
    .load()

display(df)


# write to new catalog
# we have to fetch the date--automatically-real time -not hardcoded

#import pyspark.sql.functions as F


from datetime import date
run_date = date.today().isoformat()

#run_date = '2026-09-07'

target_path = f"/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date={run_date}"

df.write.mode('overwrite')\
    .option('header','true')\
    .option('quoteAll','true')\
        .option('escape','"')\
            .csv(target_path)

# previous code ends here-------------i.e. we fetch data from sql server cloud n dump it in raw data folder of databricks

In [0]:
# New code provided by uday bhaiya in class

from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_jdbc, write_raw
import json


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("sql-server-ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")


# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)

# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]
write_options = config["write_options"]


# =================================================================
# 3.1 CREATE SCOPE N STORE ACCESS KEYS
# ================================================================

user = dbutils.secrets.get(scope="retail-platform-devv", key = source_config["user"])
password = dbutils.secrets.get(scope="retail-platform-devv", key = source_config["password"])

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat()
logger.info("Load date is %s",run_date )
logger.info("Reading data from  %s",source_config["dbtable"] )


# ============================================================
# 5. READ FROM SQL SERVER
# ============================================================
df = read_jdbc(spark, source_config["url"],source_config["dbtable"], user, password  )
logger.info("read %s rows", df.count())
logger.info("sample data.......")
df.show()


# ============================================================
# 6. TARGET PATH
# ============================================================

target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"
logger.info("writiing data to %s", target_path)

# ============================================================
# 7. WRITE RAW
# ============================================================
write_raw(df, target_path,target_config["file_format"], target_config["mode"], write_options)
logger.info("data landed at %s", target_path)




### Nilesh code starts from here

In [0]:
# I will start from here ----- homework from uday bhaiya

# here basically we fetch the data from sql server cloud and dump it in raw data folder of databricks

# we used modular architecture in a sense that we created a common functions such as read n write - which will be called for each file

# also we hide our access key using configuration - we did not hardcode it - we used databricks secret scope - to store access keys

# we also created the widgets to pass path of files

# steps are just previous steps , but this time with the use of common utils functions and confugurations - okay


# =======1.IMPORT LIBRARIES & FUNCTIONS=================

import json
from datetime import date
from common_utils.logging import get_logger
import importlib, common_utils.ingestor
importlib.reload(common_utils.ingestor)
from common_utils.ingestor import read_jdbc, write_raw

logger = get_logger('sql-server-ingestion')

# =======2.CREATE WIDGETS FOR PATH=====================
dbutils.widgets.text('path',"")
config_path = dbutils.widgets.get('path')

# =======3.CONFIG VARIABLES FOR ACCESSING KEYS USING CONFIG JSON FILE ===========

with open(config_path,'r') as f:
    config = json.load(f)

source_config = config['source']
target_config = config['target']
write_options_target = config['write_options']

print(source_config)
# here we have already created the secret scope inside the databricks n used the terminal to see it

user = dbutils.secrets.get(scope = 'retail-platform-dev', key = source_config['user'])
print(user)
password = dbutils.secrets.get(scope = 'retail-platform-dev', key= source_config['password'])
print(password)

# ========4.FETCH DATA FROM SQL SERVER & READ IT USING READ_JDBC FUNCTION=================

df = read_jdbc(spark,source_config['url'],source_config['dbtable'],user,password,driver = source_config['driver'])
logger.info('read %s rows', df.count())
logger.info('sample data looks like.....')
df.show()

# ========5.WRITING THE DATA INTO RAW DATA FOLDER WITH CURRENT DATE FOLDER ====================

from datetime import date
run_date = date.today().isoformat()
logger.info('load date is %s', run_date)


target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"
logger.info("writiing data to %s", target_path)


target_path = f"{target_config['base_path']}/{target_config['folder']}/load_date={run_date}"
logger.info('writing data to %s', target_path)

target_path = write_raw(df, target_path,target_config["file_format"], target_config["mode"], write_options_target)
logger.info("data landed at %s", target_path)
















# CODE ABOVE WORKS FINE---THIS IS GIVEN HOMEWORK----IGNORE CODE BELOW----OLD CODE

In [0]:
# we will create the widgets

dbutils.widgets.text('path',"")

In [0]:
dbutils.widgets.text('path',"")
config_path = dbutils.widgets.get('path')
print(config_path)


with open(config_path, 'r') as f:
    config = json.load(f)

print(config)

In [0]:
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_jdbc

logger= get_logger('sql-server-ingestion')

url = "jdbc:sqlserver://rivadata.database.windows.net:1433;databaseName=batch2;encrypt=true;trustServerCertificate=true;loginTimeout=90" 

user = "rivadata" 

password = dbutils.secrets.get(scope='retail-platform-dev', key='sqlserver_password')

dbtable = "retail.customers"

df = read_jdbc(url,dbtable,user,password,spark)

df.show()

# then we will write it using common utils function
# code from uday bhaiya

# step 3 --- we will pass all the parameters using config files and dictionary so that it will be useful for json data

# create config driven --then we will put all config dictionary into config folder --n then we will use access keys data from it

# =============== 1.CONFIG PATH ================================

config = {
    'source': {
        'name':'sqlserver',
        'url':"jdbc:sqlserver://rivadata.database.windows.net:1433;databaseName=batch2;encrypt=true;trustServerCertificate=true;loginTimeout=90",
        'user' : "rivadata" ,
        'password' : "<use dbutils.secrets.get()>",
        'dbtable' : "retail.customers" },

    'target': {
        'basepath':,
        ''

    },

    'write_options':
        {
            'header':
            'quoteAll':
            'escape':'"'

        }

}


# configuration ---using the config dictionary ### to access password or keep password safe

source_config = config['source']
target_config = config['target']
write_options = config['write_options']

run_date = date.today().isoformat()

url = source_config['url']
dbtable = source_config['dbtable']
user = source_config['user']
password = source_config['password']

base_path = target.config['base_path']
folder = target_config['folder']


file_format =
mode =

  





In [0]:
dbutils.library.restartPython()

In [0]:
def read_jdbc(url,dbtable,user,password,driver):
    '''
    Read a table from a relational database over JDBC

    parameters:
    url     =
    dbtable =
    user    =
    password=
    driver  =
    '''

    df = spark.read.format('jdbc')\
        .option('jdbcUrl', url)\
        .option('dbtable', 'orders')\
        .option('user', 'root')\
        .option('password', 'password')\
        .option('driver', 'org.postgresql.Driver')\
        .load()

In [0]:

url = "jdbc:sqlserver://rivadata.database.windows.net:1433;databaseName=batch2;encrypt=true;trustServerCertificate=true;loginTimeout=90" 

user = "rivadata" 

password = dbutils.secrets.get(scope='retail-platform-dev', key='sqlserver_password')

dbtable = "retail.customers"

df = spark.read.format('jdbc')\
    .option('url',url)\
    .option('dbtable',dbtable)\
    .option('user', user)\
    .option('password', password)\
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')\
    .load()

display(df)



In [0]:
# write to new catalog
# we have to fetch the date--automatically-real time -not hardcoded

#import pyspark.sql.functions as F


from datetime import date
run_date = date.today().isoformat()

#run_date = '2026-09-07'

target_path = f"/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date={run_date}"

df.write.mode('overwrite')\
    .option('header','true')\
    .option('quoteAll','true')\
        .option('escape','"')\
            .csv(target_path)

In [0]:
# read data from raw_data volume - done
# add audit columnms -> 2 columns - last_update_ts, file_path
# define target
# write data in delta table


#logging
'''
basic - print statement

advanced - python logger

'''

import pyspark.sql.functions as F 


print("Defining raw path")
raw_path = "/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date=2026-09-07/"
print(f"raw path given is {raw_path}")


print("reading data")
df = spark.read\
        .format("csv")\
        .option("header","true")\
        .option("inferSchema", "true")\
        .load(raw_path)

print("sample data",df.show())

print("adding audit columns")
df = df.withColumn("last_update_ts", F.current_timestamp() )\
        .withColumn("file_path", F.col("_metadata.file_path"))

target_path = "retaildataplatform.bronze.sqlserver_customers"
print(f"target path is {target_path}")

print("writing data")
df.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(target_path)

print("Data writeen Successfully at", target_path)
